In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpam5h74hq".


In [4]:
%%cuda
#include <iostream>

// THE KERNEL
__global__ void filterEven(float* data, int n) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;

    if (i < n) {
        // Cast to int to use the modulo operator
        if ((int)data[i] % 2 != 0) {
            data[i] = 0.0f; // Set odd numbers to zero
        }
        // Even numbers remain untouched
    }
}

int main() {
    int n = 10;
    size_t size = n * sizeof(float);

    float *h_data = (float*)malloc(size);

    for(int i=0; i < n; i++) h_data[i] = (float)i; // [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

    float *d_data;
    cudaMalloc(&d_data, size);
    cudaMemcpy(d_data, h_data, size, cudaMemcpyHostToDevice);

    filterEven<<<1, 256>>>(d_data, n);

    cudaMemcpy(h_data, d_data, size, cudaMemcpyDeviceToHost);

    std::cout << "Original: 0 1 2 3 4 5 6 7 8 9" << std::endl;
    std::cout << "Filtered: ";
    for(int i=0; i < n; i++) std::cout << h_data[i] << " ";
    std::cout << std::endl;

    cudaFree(d_data);
    free(h_data);
    return 0;
}

Original: 0 1 2 3 4 5 6 7 8 9
Filtered: 0 0 2 0 4 0 6 0 8 0 

